<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/NLP/01-core-components.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to NLP guideline](Natural-Language-Processing.html)


## **Foundations and Core Components of NLP Systems** {#foundations-and-core-components-of-nlp-systems}

An NLP system transforms human language into a decision, structured annotation, ranked result, or newly generated text. The visible model is only one part of that process. Before choosing an architecture, we must decide what linguistic phenomenon matters, how it will be represented in data, what output the system should produce, how success will be measured, and how the parameters will be learned.

A useful high-level view is:

$$
\text{language in context}
\rightarrow \text{data and representation}
\rightarrow \text{model}
\rightarrow \text{inference}
\rightarrow \text{output}
\rightarrow \text{evaluation}
$$

This pipeline is not strictly one-way. Evaluation errors often reveal a weakness in the annotation scheme, data coverage, tokenisation, or task definition rather than a weakness in the model alone. The purpose of this chapter is therefore to establish two foundations: **how language is structured** and **how an NLP problem becomes a learnable system**.

### **Linguistic Foundations** {#linguistic-foundations}

Human language is organized at several interacting levels. Morphology studies the internal structure of words; syntax studies how words form phrases and sentences; semantics studies literal meaning; pragmatics studies meaning in use; and discourse studies how meaning develops across sentences or turns. These levels are not merely theoretical categories. They determine what information an NLP system must preserve and what kinds of mistakes it can make.

For example, consider the message: *"Great, the update deleted my files again."* A tokenizer can identify the words, a parser can identify the grammatical relations, and a sentiment lexicon may treat *great* as positive. Correct interpretation, however, requires pragmatic knowledge: the contradiction between *great* and the negative event signals sarcasm. A system that only recognizes words but ignores context will confidently produce the wrong answer.

#### **Morphology** {#morphology}

**Morphology** studies how words are built from smaller meaning-bearing units called **morphemes**. In *unhelpfulness*, the prefix *un-* expresses negation, *help* carries the central lexical meaning, *-ful* forms an adjective, and *-ness* forms a noun. Morphemes are not always independent words, but they systematically modify meaning or grammatical function.

Two distinctions are especially useful:

| Process | Purpose | Example | NLP relevance |
|---|---|---|---|
| Inflection | Changes grammatical form without creating a new lexeme | *walk* $\rightarrow$ *walked*, *cat* $\rightarrow$ *cats* | Lemmatization, agreement, tense, and feature prediction |
| Derivation | Creates a new word or changes its grammatical category | *happy* $\rightarrow$ *unhappy*, *teach* $\rightarrow$ *teacher* | Vocabulary growth, sentiment, and compositional meaning |
| Compounding | Combines lexical units | *credit card*, *smartphone* | Multiword expressions and token boundaries |

Morphology matters because treating every surface form as unrelated wastes evidence. If *connect*, *connected*, and *connecting* share a root, a model can generalize across forms. This is especially important for morphologically rich languages, where one lemma may produce many surface forms, and for low-resource settings, where each form may appear only a few times. Subword tokenisation later offers a data-driven way to recover some of this internal structure, but linguistic morphemes and model subwords are not necessarily identical.

#### **Syntax** {#syntax}

**Syntax** describes how words combine into larger structures. Two common representations are **constituency structure**, which groups words into nested phrases, and **dependency structure**, which represents directed relations between a head word and its dependents.

In *"The analyst reviewed the report,"* the verb *reviewed* is the structural center. *Analyst* is its subject, *report* is its object, and both nouns have determiner dependents. These relations help answer *who did what to whom*. Word order alone is not always sufficient, particularly in long sentences or languages with flexible word order.

Syntactic information supports information extraction, relation extraction, question answering, translation, and grammatical error detection. It can also expose attachment ambiguity. In *"I saw the researcher with a telescope,"* the phrase *with a telescope* may describe the instrument used for seeing or the researcher being observed. Different parse structures lead to different interpretations.

#### **Semantics** {#semantics}

**Semantics** concerns the literal meaning of words, phrases, and sentences. **Lexical semantics** studies relationships among word meanings, such as synonymy, antonymy, and word senses. **Compositional semantics** asks how the meanings of parts combine under a syntactic structure to form the meaning of the whole.

The word *bank* may denote a financial institution or the side of a river. Selecting the intended sense requires context. Sentence meaning also depends on roles: in *"The storm damaged the roof,"* *storm* is the cause and *roof* is the affected entity. Tasks such as word-sense disambiguation, semantic role labeling, natural language inference, and question answering attempt to model different aspects of this layer.

Semantic similarity should not be confused with logical equivalence. *"A dog is running"* and *"An animal is moving"* are similar and the first normally entails the second, but the reverse does not hold. An embedding can place them close together while still failing to preserve that directional relationship.

#### **Pragmatics** {#pragmatics}

**Pragmatics** studies what a speaker intends to communicate in a particular situation. The literal sentence is interpreted together with speaker identity, shared knowledge, social conventions, location, time, and conversational goals.

When someone asks *"Can you open the window?"*, the literal form is a question about ability, but the usual communicative act is a request. Expressions such as *I*, *here*, *today*, and *that one* are **deictic**: their referents depend on the situation. Irony, politeness, indirect requests, presupposition, and conversational implicature are also pragmatic phenomena.

Pragmatics is central to dialogue systems, affective computing, moderation, and intent detection. It is also difficult to annotate because two readers may reasonably infer different intentions when contextual information is missing. A model trained only on isolated sentences can therefore learn dataset shortcuts while missing the actual communicative meaning.

#### **Discourse** {#discourse}

**Discourse** extends analysis beyond a single sentence. It studies how sentences and conversational turns form a coherent whole through reference, topic progression, temporal order, and rhetorical relations such as cause, contrast, and elaboration.

Consider: *"Maya submitted the paper. She revised it twice before the deadline."* Understanding the second sentence requires resolving *she* to *Maya* and *it* to *the paper*. In longer documents, a system must also track entities, events, speaker turns, and which information is new or already established.

Discourse modeling supports document summarization, long-document question answering, coreference resolution, dialogue-state tracking, and multi-turn assistants. A sentence-level model may perform well on local benchmarks yet contradict itself across paragraphs because it lacks a stable representation of the broader discourse.

#### **Ambiguity** {#ambiguity}

Natural language frequently permits more than one interpretation. Ambiguity is therefore not noise that can always be removed; it is a property that the model may need to represent and resolve using context.

| Ambiguity type | Example | Competing interpretations |
|---|---|---|
| Lexical | *I went to the bank.* | Financial institution or river bank |
| Morphological | *unlockable* | Cannot be locked, or capable of being unlocked |
| Syntactic | *I saw the researcher with a telescope.* | Instrument attachment or noun-phrase attachment |
| Scope | *Every student read a book.* | One shared book or possibly a different book per student |
| Referential | *Lena told Priya that she won.* | *She* may refer to Lena or Priya |
| Pragmatic | *That was a brilliant decision.* | Sincere praise or sarcasm |

The appropriate response depends on the application. A search engine may preserve several interpretations and retrieve diverse results. A voice assistant may ask a clarification question. A classifier must usually commit to one label, while a probabilistic model can express uncertainty through a distribution such as $p(y\mid x)$. This is one reason NLP predictions should not automatically be treated as certain facts.

The following example uses spaCy to inspect several linguistic layers produced for the same sentence. `lemma_` approximates the dictionary form, `morph` exposes inflectional features, `dep_` gives the dependency relation, and `head` identifies the syntactic governor. The dependency visualization is generated locally in the notebook, so it does not rely on a fragile externally hosted image. See the official [spaCy linguistic features](https://spacy.io/usage/linguistic-features) and [visualizer documentation](https://spacy.io/usage/visualizers/) for the underlying annotations and display options.

<details>
<summary>Python: Inspecting morphology and syntax with spaCy</summary>

```python
import spacy
from spacy import displacy

# Install once if needed:
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

text = "The researchers were analyzing ambiguous sentences."
doc = nlp(text)

# Inspect token-level lexical, morphological, and syntactic information.
for token in doc:
    print({
        "token": token.text,
        "lemma": token.lemma_,
        "part_of_speech": token.pos_,
        "morphology": str(token.morph),
        "dependency": token.dep_,
        "head": token.head.text,
    })

# In a Jupyter notebook, this displays an interpretable dependency diagram.
displacy.render(doc, style="dep", jupyter=True, options={"compact": True})
```
</details>

The output is useful as an analysis aid, but it should not be mistaken for perfect linguistic truth. The annotations are model predictions, and their errors may propagate into downstream rules or features. Modern end-to-end systems often learn contextual representations without explicitly consuming a parse tree, yet the same linguistic distinctions remain valuable for designing datasets, diagnosing errors, and explaining what the system has failed to understand.

### **Data** {#data}

In NLP, **data** is a collection of language observations together with the context and supervision needed to learn or evaluate a task. A row of text is not automatically a useful training example. Its value depends on where it came from, what population and time period it represents, how it was cleaned, what labels were attached, and whether those labels match the intended application.

The required scale varies dramatically. A narrow sentiment classifier may begin with thousands of labeled reviews, machine translation may use millions of sentence pairs, and a general-purpose language model may be pretrained on billions or trillions of tokens. More data can improve coverage, but quantity cannot repair a mismatched task definition, systematic label errors, privacy violations, or leakage between training and evaluation sets.

#### **Corpora and Data Sources** {#corpora-and-data-sources}

A **corpus** is a deliberately collected body of language data. Corpora may be general-domain or specialized, monolingual or multilingual, written or spoken, static or continuously updated. The source determines which vocabulary, dialects, styles, and social groups are visible to the model.

| Source | Typical strengths | Typical risks |
|---|---|---|
| News and edited publications | Grammatical, information-rich text | Formal style, editorial and geographic bias |
| Social media | Current language, slang, interaction signals | Noise, bots, privacy concerns, demographic skew |
| Product or service reviews | Natural sentiment and aspect signals | Selection bias, fake reviews, rating-text mismatch |
| Conversation transcripts | Turn-taking, intent, pragmatic context | Sensitive information, transcription errors |
| Domain documents | Precise legal, medical, or scientific terminology | Restricted access, narrow distribution, expert annotation cost |
| Synthetic data | Controllable labels and rare-case coverage | Generator artifacts and reduced linguistic diversity |

**Representativeness** means that the dataset covers the language situations in which the system will operate. A model trained on carefully edited English news cannot be assumed to handle code-switching, informal Australian English, or clinical notes. The correct question is not simply *"Is the dataset large?"* but *"Large and representative of what?"*

Corpus documentation should record provenance, collection dates, languages, licenses, consent assumptions, known exclusions, filtering steps, and intended uses. These details make later error analysis possible and prevent a convenient dataset from silently becoming the wrong benchmark.

#### **Annotation and Label Design** {#annotation-and-label-design}

Supervised NLP requires an **annotation scheme** that translates a conceptual phenomenon into observable labels. Before annotation begins, the designer must define the unit of analysis, the label inventory, boundary rules, treatment of uncertainty, and examples of difficult cases.

For sentiment analysis, the unit might be an entire review, a sentence, or a particular aspect such as *battery life*. The labels might be binary, three-way, ordinal, continuous, or multi-label. These choices create different tasks even when the underlying text is identical. For named entity recognition, annotators must additionally agree on span boundaries and whether expressions such as nationalities, products, or fictional locations count as entities.

A practical annotation workflow is:

$$
\text{task definition}
\rightarrow \text{pilot annotation}
\rightarrow \text{guideline revision}
\rightarrow \text{independent annotation}
\rightarrow \text{agreement analysis}
\rightarrow \text{adjudication}
$$

Independent labels reveal whether the instructions are reproducible. **Cohen's kappa** measures agreement between two annotators while correcting for agreement expected by chance:

$$
\kappa = \frac{p_o-p_e}{1-p_e}
$$

Here, $p_o$ is the observed proportion of examples on which the annotators agree, and $p_e$ is the agreement expected if each annotator independently followed their own label frequency. The numerator $p_o-p_e$ removes chance agreement; the denominator $1-p_e$ scales the result relative to the maximum improvement available beyond chance. A value near $1$ indicates strong agreement, $0$ indicates agreement close to chance, and a negative value indicates systematic disagreement. The interpretation must still consider task difficulty and label prevalence, because kappa can be unstable when one class dominates.

<details>
<summary>Python: Measuring and inspecting annotation agreement</summary>

```python
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Two annotators independently label the same sentiment examples.
annotator_a = ["positive", "negative", "neutral", "positive", "negative", "neutral"]
annotator_b = ["positive", "neutral",  "neutral", "positive", "negative", "positive"]

# Kappa summarizes chance-corrected agreement.
kappa = cohen_kappa_score(annotator_a, annotator_b)
print(f"Cohen's kappa: {kappa:.3f}")

# The confusion matrix shows which label distinctions caused disagreement.
labels = ["negative", "neutral", "positive"]
matrix = confusion_matrix(annotator_a, annotator_b, labels=labels)
print("Rows = annotator A; columns = annotator B")
print(matrix)

# In a real project, review the disagreement cases and revise guidelines
# before adjudicating them into a final gold label.
disagreements = [
    (index, a, b)
    for index, (a, b) in enumerate(zip(annotator_a, annotator_b))
    if a != b
]
print("Disagreements:", disagreements)
```
</details>

Agreement is diagnostic rather than ceremonial. Frequent disagreement between *neutral* and *negative* may indicate vague definitions, insufficient context, or a genuinely subjective boundary. Adjudication can produce a final reference label, but it should not erase the uncertainty revealed during annotation. For subjective tasks, retaining label distributions or annotator metadata may be more faithful than forcing every example into a single supposedly objective answer.

#### **Data Quality** {#data-quality}

Data quality has several dimensions: **correctness** of text and labels, **coverage** of relevant cases, **consistency** of formatting and annotation, **uniqueness** of examples, **timeliness**, **privacy**, and **balance** across labels and user groups. Each dimension creates a different failure mode.

Common NLP data problems include:

- exact duplicates and near-duplicates that overweight repeated content;
- train-test leakage from the same article, user, conversation, or template;
- incorrect encodings, truncated documents, markup, and language-identification errors;
- label noise, ambiguous instructions, and annotator shortcuts;
- class imbalance that makes accuracy look high while minority cases are ignored;
- temporal mismatch, such as evaluating a current system on outdated vocabulary;
- private, copyrighted, toxic, or identifying content collected without adequate controls;
- benchmark contamination, where evaluation examples occur in pretraining or tuning data.

Cleaning must preserve the phenomenon being modeled. Removing punctuation may harm emotion recognition; lowercasing may erase named-entity cues; deleting emojis may remove the strongest sentiment signal; and aggressive deduplication may discard legitimate recurring formulas in technical text. Every transformation should therefore be justified by the task rather than applied as a universal recipe.

<details>
<summary>Python: A compact text-dataset quality audit</summary>

```python
import re
import pandas as pd

data = pd.DataFrame({
    "text": [
        "Great battery life!",
        "great  battery life!",       # normalized near-duplicate
        None,                          # missing text
        "Terrible support 😞",
        "Click https://example.com",  # possible template or promotional text
    ],
    "label": ["positive", "positive", "negative", "negative", "positive"],
})

def normalize_for_audit(text):
    # Create an audit key without changing the original training text.
    if pd.isna(text):
        return None
    text = text.lower().strip()
    return re.sub(r"\s+", " ", text)

# 1. Check missing values before tokenization or model training.
print("Missing values:\n", data.isna().sum())

# 2. Inspect label balance; do not rely on total dataset size alone.
print("\nLabel counts:\n", data["label"].value_counts(dropna=False))

# 3. Detect normalized duplicates while retaining the original text.
data["audit_key"] = data["text"].map(normalize_for_audit)
duplicate_rows = data[data.duplicated("audit_key", keep=False)]
print("\nPotential duplicates:\n", duplicate_rows)

# 4. Add task-specific checks instead of silently deleting rows.
data["contains_url"] = data["text"].fillna("").str.contains(r"https?://", regex=True)
data["character_count"] = data["text"].fillna("").str.len()
print("\nRows requiring review:\n", data[data["contains_url"] | data["text"].isna()])
```
</details>

An audit should produce a review report, not an automatic deletion list. Suspicious examples need to be traced back to their source and assessed against the intended use. Otherwise, cleaning can replace visible noise with invisible sampling bias.

#### **Dataset Construction and Splitting** {#dataset-construction-and-splitting}

Dataset construction turns a corpus into train, validation, and test examples. The **training set** is used to estimate model parameters. The **validation set** supports model selection, threshold selection, and hyperparameter tuning. The **test set** should remain untouched until the final evaluation so that it estimates performance on unseen data rather than performance after repeated adaptation to the benchmark.

Random row-level splitting is appropriate only when rows are sufficiently independent and identically distributed. NLP datasets often violate this assumption. Reviews from the same author share style, messages in one conversation share context, sentences from one article share content, and templated documents share long spans of text. If related examples cross the split boundary, the model can appear to generalize while merely recognizing a source or template.

| Deployment question | Appropriate split | Leakage prevented |
|---|---|---|
| Will the model handle unseen documents from familiar sources? | Group by document | Sentence overlap across one document |
| Will it handle new users or speakers? | Group by user or speaker | Author-specific style and identity cues |
| Will it handle new conversations? | Group by conversation | Shared dialogue history across splits |
| Will it work on future language? | Chronological split | Future information appearing in training |
| Will it transfer to a new domain? | Domain-held-out test set | Domain vocabulary memorization |

Stratification can preserve label proportions, but it does not by itself prevent group leakage. When both constraints matter, grouping should be applied first and label balance checked afterward. The final test set should also contain meaningful slices, such as language variety, document length, rare labels, or negation, so that aggregate performance does not hide a critical weakness.

The distinction from the later evaluation chapter is important: this section asks **how trustworthy data is constructed**, while evaluation asks **which measurements and experimental protocols demonstrate system behavior**. Good metrics cannot rescue a contaminated split, and a clean split cannot rescue a metric that ignores the real cost of errors.

#### **From Language Phenomena to Data Design** {#from-language-phenomena-to-data-design}

The linguistic target should determine the annotation unit and the data collection strategy. The table below summarizes the connection.

| Phenomenon | Suitable data unit | Possible supervision | Typical design risk |
|---|---|---|---|
| Morphology | Token or morpheme | Lemma and feature tags | Treating model subwords as true morphemes |
| Syntax | Sentence | Constituency or dependency trees | Parser conventions treated as universal grammar |
| Semantics | Word, span, sentence, or pair | Sense, role, similarity, entailment | Similarity confused with factual equivalence |
| Pragmatics | Utterance plus situation | Intent, stance, sarcasm, dialogue act | Missing speaker and context information |
| Discourse | Document or conversation | Coreference, relation, dialogue state | Breaking connected text into isolated rows |

The central lesson is that data design is already a form of modeling. Choosing the unit, label, context window, source population, and split encodes assumptions about what language means and what the system should learn. A sophisticated architecture trained on poorly framed data will optimize the wrong problem more efficiently.


### **Model** {#model}
$$\operatorname{score}(x, y)$$
A function that maps (input, output) pairs to scores. In most tasks(supervised), model is the map relationship between the input(document) and output(label) from the data. There are four common types of **Model** in NLP System:

- **Rule-based Model**(1966-1980s): it relies on **manually designed linguistic rules** to process language. For example "If the review contains "good" return positive". Which has strong interpretability, but with non-extensible and high maintenance costs.
- **Statistical Model**(1990s): describe language phenomena using probabilistic models, for example: n-gram Language Model, Naive Bayes, Hidden Markov Model(HMM). These models estimate probabilities from data and were widely used in early NLP systems.
- **Classical Machine Learning**(1990s): it treats NLP as a **feature-based learning problem**. Text is first converted into features (e.g., Bag-of-Words or TF-IDF), and then a machine learning model is trained.For example: Logistic Regression, Linear Model, Support Vector Machine(SVM), Decision Tree. These models are efficient and often serve as strong baselines.
- **Neural Network Model**(2010s): it learns **distributed representations** of text and extract features automatically through multiple layers., like Feedforward Network, Recurrent Neural Network(RNN), Convolutional Neural Network (CNN), Long Short-Term Memory (LSTM). These models can capture complex patterns and contextual relationships in text.
- **Transformer-based Model**(2018-): modern NLP systems are largely based on **Transformer architectures**. Transformers convert text into vector representations and use **self-attention mechanisms** to model relationships between words in a sequence.  
Deep neural layers then learn contextual semantic representations, which can be used for prediction or text generation. For instances, Bidirectional Encoder Representations from Transformers(BERT), Generative Pre-trained Transformer(GPT).

### **Inference Method** {#inference-method}
A way to make a prediction for an example given a **Model**. It finds an output that a model gives a high score to (not necessarily the highest scoring one and not necessarily the true best).
$$
y^* = \arg\max_{y} score(x, y)
$$
<details>
    
<summary>Simple approach</summary>
    
```python
best_score = -inf
best_label = None

for label in labels:
    score = model(x, label)
    if score > best_score:
        best_score = score
        best_label = label

return best_label
```
</details>

**Common Inference Algorithms**:

- **Greedy Search**: it selects the best option at each step. Fast and simple, but may miss the global optimum because it only considers local decisions.
- **Beam Search**: it keeps the top k candidates at each step, which produces better results than greedy search, but with higher computational cost.
- **A-star Search**: it is a heuristic search algorithm that explores the most promising paths first. It is useful when the search space is large but structured.
- **Dynamic Programming**(e.g. Viterbi Search): it is used when problems contain overlapping subproblems. It improves efficiency by avoiding repeated computation. 

### **Metric** {#metric}
A function that gives a score to the output produced by a **Model** given some **Data**. It gives a measurement of how good the output of your system is compared to the best answer. For evaluation metrics, please refer to Statistic/Modeling/Model Evaluation part

**Common Metric**:

- **0-1 Loss**: the simplest evaluation metric for classification. Very intuitive, directly reflects classification errors, but not continuous, differentiable, difficult to optimize using gradient-based methods. Therefore, 0-1 loss is mostly used for theoretical evaluation rather than training models. 
    - $L(y,f(x)) =\begin{cases} 0 & \text{if prediction is correct} \\ 1 & \text{otherwise}\end{cases}$
    - Correct prediction → loss = 0. Incorrect prediction → loss = 1.

- **Hinge Loss**($L = \max(0, 1 - y \cdot f(x))$, where: $y$ is the true label (usually $+1$ or $-1$), $f(x)$ is the model output): a prediction should not only be correct, but should also have a sufficient margin. Hinge loss is commonly used in **Support Vector Machines (SVM)**.
    - Confident correct prediction → loss = 0
    - Prediction near the boundary → small loss
    - Wrong prediction → large loss

- **Cross-Entropy Loss**($L = -\log p(y)$): is one of the **most widely used metrics in NLP**. If the model assigns high probability to the correct label, the loss will be small.
    - For multi-class classification:$L = -\sum_{i \in classes} y_i \log(p(y_i))$
    - where: $y_i$ is the true label indicator, $p(y_i)$ is the predicted probability

- **Squared Error**($L = (y - f(x))^2$): it measures the squared difference between the true value and the predicted value. The larger the difference between prediction and true value, the larger the loss.

### **Learning Method** {#learning-method}
A way to update a **Model** given **Data**, a **Metric**, and an **Inference Method**. It is an algorithm that uses data and a metric to see what mistakes your system makes and then updates the model to reduce those mistakes. The learning algorithm updates the parameters iteratively in order to reduce the loss. Formally, the learning process can be written as: $\min_{\theta} L(\theta)$, where: $\theta$ represents the model parameters, $L(\theta)$ is the loss function

**Learning Process**: Input data -> Model prediction -> Compute loss -> Update parameters -> Repeat

**Common Learning Method**(Optimizer ):

- **Gradient Descent**($\theta = \theta - \eta \nabla L(\theta)$): it computes gradients using the **entire dataset** and updates parameters in the direction that reduces the loss. Because it uses all data for each update, it provides stable and accurate gradient estimates but becomes very slow when datasets are large. Its main advantages are simplicity and stable convergence, making it useful for theoretical analysis and small datasets. However, it is computationally expensive and impractical for large-scale deep learning.

- **Stochastic Gradient Descent(SGD, $\theta = \theta - \eta \nabla L_i(\theta)$, where $L_i$ is the loss for one sample or mini-batch.)**: it improves efficiency by updating parameters using **one sample or a small mini-batch** at a time instead of the entire dataset. This greatly speeds up training and allows models to scale to large datasets. The tradeoff is that the updates are noisy and can oscillate around the optimal solution. SGD works well in large-scale neural network training but may require careful tuning of the learning rate.

- **Adam**(Adaptive Moment Estimation): Adam combines the ideas of **Momentum and RMSProp** by maintaining both the first moment (mean of gradients) and second moment (variance of gradients). This allows it to adapt learning rates automatically while also smoothing gradient updates. Adam is one of the most widely used optimizers in deep learning and NLP because it converges quickly and requires minimal tuning. However, it can sometimes produce slightly worse generalization than SGD in some tasks.

    - First moment: $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$
    - Second moment: $v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$
    - Parameter update: $\theta = \theta - \eta \frac{m_t}{\sqrt{v_t} + \epsilon}$



<details>
<summary>NLP System Example</summary>

```python
import re
import random
from collections import Counter
from typing import List, Tuple, Dict

# ============================================================
# 1. DATA
# ============================================================
# Data = examples of the language phenomena we want the system to handle
# 这里我们用一个小型情感分类数据集，每条数据是 (text, label)
# label: 1 = positive, 0 = negative
# ============================================================

dataset: List[Tuple[str, int]] = [
    ("this movie is fantastic", 1),
    ("i love this film", 1),
    ("what a great and wonderful story", 1),
    ("this was an amazing performance", 1),
    ("i really enjoyed this movie", 1),
    ("the acting was brilliant", 1),
    ("this film is very good", 1),
    ("absolutely loved the ending", 1),
    ("the movie was inspiring and touching", 1),
    ("a delightful and enjoyable film", 1),

    ("this movie is terrible", 0),
    ("i hate this film", 0),
    ("what a boring and awful story", 0),
    ("this was a horrible performance", 0),
    ("i really disliked this movie", 0),
    ("the acting was terrible", 0),
    ("this film is very bad", 0),
    ("absolutely hated the ending", 0),
    ("the movie was dull and disappointing", 0),
    ("a painful and unpleasant film", 0),
]

random.seed(42)
random.shuffle(dataset)

split = int(0.8 * len(dataset))
train_data = dataset[:split]
test_data = dataset[split:]


# ============================================================
# Text preprocessing (still part of DATA representation)
# ============================================================
# 这里做最简单的 tokenization：小写 + 按单词切分
# ============================================================

def tokenize(text: str) -> List[str]:
    return re.findall(r"\b\w+\b", text.lower())


def build_vocab(data: List[Tuple[str, int]]) -> Dict[str, int]:
    vocab = {}
    for text, _ in data:
        for token in tokenize(text):
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab


vocab = build_vocab(train_data)


def vectorize(text: str, vocab: Dict[str, int]) -> List[float]:
    """
    Bag-of-Words representation:
    text -> feature vector
    """
    vec = [0.0] * len(vocab)
    counts = Counter(tokenize(text))
    for token, count in counts.items():
        if token in vocab:
            vec[vocab[token]] = float(count)
    return vec


X_train = [vectorize(text, vocab) for text, _ in train_data]
y_train = [label for _, label in train_data]

X_test = [vectorize(text, vocab) for text, _ in test_data]
y_test = [label for _, label in test_data]


# ============================================================
# 2. MODEL
# ============================================================
# Model = a function that maps (input, output) pairs to scores
# 这里我们用最简单的线性模型：
# score(x) = w · x + b
# 如果 score >= 0，倾向 positive
# 如果 score < 0，倾向 negative
# ============================================================

class LinearSentimentModel:
    def __init__(self, num_features: int):
        self.weights = [0.0] * num_features
        self.bias = 0.0

    def score(self, x: List[float]) -> float:
        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias


# ============================================================
# 3. INFERENCE METHOD
# ============================================================
# Inference Method = a way to make a prediction given a model
# 这里的推理规则是：
# pred = 1 if score >= 0 else 0
# ============================================================

def predict(model: LinearSentimentModel, x: List[float]) -> int:
    score = model.score(x)
    return 1 if score >= 0 else 0


# ============================================================
# 4. LEARNING METHOD
# ============================================================
# Learning Method = a way to update the model using data + errors
# 这里我们实现 Perceptron Learning：
#
# 如果预测正确，不更新
# 如果预测错误：
#   positive(1) but predicted negative(0): w = w + x, b = b + 1
#   negative(0) but predicted positive(1): w = w - x, b = b - 1
#
# 为了方便，我们把 label 转成:
#   positive -> +1
#   negative -> -1
#
# then:
#   if y * score <= 0:
#       w = w + y * x
#       b = b + y
# ============================================================

def train_perceptron(
    model: LinearSentimentModel,
    X: List[List[float]],
    y: List[int],
    epochs: int = 10
) -> None:
    """
    Train using the perceptron update rule.
    """
    y_signed = [1 if label == 1 else -1 for label in y]

    for epoch in range(epochs):
        mistakes = 0
        for x, y_true in zip(X, y_signed):
            score = model.score(x)

            # mistake or boundary case
            if y_true * score <= 0:
                mistakes += 1
                for i in range(len(model.weights)):
                    model.weights[i] += y_true * x[i]
                model.bias += y_true

        print(f"Epoch {epoch + 1}: mistakes = {mistakes}")


# ============================================================
# 5. METRIC
# ============================================================
# Metric = a function that measures how good predictions are
# 这里我们用 Accuracy
# ============================================================

def accuracy(model: LinearSentimentModel, X: List[List[float]], y: List[int]) -> float:
    correct = 0
    for x, y_true in zip(X, y):
        y_pred = predict(model, x)
        if y_pred == y_true:
            correct += 1
    return correct / len(y)


# ============================================================
# Run the full NLP system
# ============================================================

model = LinearSentimentModel(num_features=len(vocab))

print("Before training:")
print("Train Accuracy:", accuracy(model, X_train, y_train))
print("Test Accuracy :", accuracy(model, X_test, y_test))

print("\nTraining...")
train_perceptron(model, X_train, y_train, epochs=10)

print("\nAfter training:")
print("Train Accuracy:", accuracy(model, X_train, y_train))
print("Test Accuracy :", accuracy(model, X_test, y_test))


# ============================================================
# Demo predictions
# ============================================================

demo_texts = [
    "this movie was wonderful",
    "i hated this boring film",
    "the acting was good",
    "the ending was awful",
    "what an inspiring story",
]

print("\nDemo predictions:")
for text in demo_texts:
    x = vectorize(text, vocab)
    score = model.score(x)
    label = predict(model, x)
    sentiment = "positive" if label == 1 else "negative"
    print(f"Text: {text}")
    print(f"Score: {score:.2f}, Prediction: {sentiment}")
    print("-" * 50)
```
</details>

